# Difference-in-Differences: Netflix Recommendation Algorithm Launch

## Business Problem

Netflix launches a new recommendation algorithm in **10 of 20 regions**. The rollout happens at **month 12** -- the first 10 regions get the update, while the remaining 10 continue with the old algorithm.

**Question**: Does the new recommendation engine causally increase daily watch hours?

We can't simply compare watch hours in treated vs. untreated regions after the launch, because regions may differ in baseline viewing behavior. DiD solves this by comparing the *change* in watch hours across the two groups, removing any time-invariant differences.

### Why DiD is the right method here
- We have **panel data**: 20 regions observed over 24 months
- There is a **clear intervention point**: month 12
- We have **treated and control groups**: 10 regions each
- The setting is a **regional rollout**, not user-level randomization, so we use a quasi-experimental design

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
from matplotlib.lines import Line2D

np.random.seed(42)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

## Step 1: Simulate Panel Data

**WHY**: We need a dataset with regions observed across months, a clear treatment date, and a known true effect so we can verify our estimator recovers it.

### Data Generating Process

Each observation is a region-month with daily watch hours determined by:

1. **Region fixed effects**: Each region has a different baseline level of watch hours (e.g., Region A averages 4.5 hrs/day, Region B averages 5.2 hrs/day). These permanent differences are exactly what DiD differences out.
2. **Common time trend**: All regions experience a slight upward trend in watch hours over time (reflecting general platform growth). This is the trend both groups share.
3. **Treatment effect**: Starting at month 12, treated regions receive a boost of **+0.8 hours/day**. This is the causal effect we want to recover.
4. **Noise**: Random variation at the region-month level.

```
watch_hours = region_baseline + 0.05 * month + 0.8 * (treated & post) + noise
```

In [ ]:
n_regions = 20
n_months = 24
treatment_month = 12
true_effect = 0.8

region_baselines = np.random.uniform(3.5, 6.0, size=n_regions)
treated_regions = list(range(10))  # regions 0-9 are treated
control_regions = list(range(10, 20))  # regions 10-19 are control

rows = []
for region in range(n_regions):
    is_treated = 1 if region in treated_regions else 0
    for month in range(1, n_months + 1):
        is_post = 1 if month > treatment_month else 0
        watch_hours = (
            region_baselines[region]
            + 0.05 * month
            + true_effect * is_treated * is_post
            + np.random.normal(0, 0.3)
        )
        rows.append({
            "region": f"R{region:02d}",
            "month": month,
            "treated": is_treated,
            "post": is_post,
            "watch_hours": watch_hours,
        })

df = pd.DataFrame(rows)
df["treated_post"] = df["treated"] * df["post"]

print(f"Dataset: {len(df)} observations ({n_regions} regions x {n_months} months)")
print(f"Treated regions: {len(treated_regions)}, Control regions: {len(control_regions)}")
print(f"Treatment month: {treatment_month}, True effect: {true_effect} hrs/day")
print()
df.head(10)

## Step 2: Naive Comparison (Why It Fails)

The simplest approach is to compare average watch hours in treated vs. control regions after the launch. But this ignores the fact that regions had **different baseline levels** before the intervention.

If treated regions happened to have higher baseline watch hours, the naive comparison would overstate the effect. If they had lower baselines, it would understate it. The naive estimate conflates the treatment effect with pre-existing group differences.

In [ ]:
# Naive comparison: post-period means
post_treated = df[(df["treated"] == 1) & (df["post"] == 1)]["watch_hours"].mean()
post_control = df[(df["treated"] == 0) & (df["post"] == 1)]["watch_hours"].mean()
naive_estimate = post_treated - post_control

print(f"Post-period mean (treated):  {post_treated:.3f} hrs/day")
print(f"Post-period mean (control):  {post_control:.3f} hrs/day")
print(f"Naive estimate:              {naive_estimate:.3f} hrs/day")
print(f"True effect:                 {true_effect:.3f} hrs/day")
print(f"Bias:                        {naive_estimate - true_effect:.3f} hrs/day")
print()
print("The naive estimate is biased because it includes pre-existing differences between groups.")

# Time series plot showing the problem
monthly_means = df.groupby(["month", "treated"])["watch_hours"].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 6))
for grp, label, color in [(1, "Treated regions", "#e74c3c"), (0, "Control regions", "#3498db")]:
    data = monthly_means[monthly_means["treated"] == grp]
    ax.plot(data["month"], data["watch_hours"], marker="o", label=label, color=color, linewidth=2)

ax.axvline(x=treatment_month + 0.5, color="gray", linestyle="--", linewidth=1.5, label="Treatment (month 12)")
ax.set_xlabel("Month")
ax.set_ylabel("Average Daily Watch Hours")
ax.set_title("Treated vs. Control Regions Over Time")
ax.legend()
plt.tight_layout()
plt.show()

print("\nNotice: the gap between groups existed BEFORE the treatment.")
print("The naive comparison captures this pre-existing gap + the true treatment effect.")

## Step 3: Why DiD and Not Other Methods?

Before applying DiD, it's important to justify the method choice by considering alternatives:

| Method | Why NOT for this problem |
|---|---|
| **PSM** | PSM compares units cross-sectionally and ignores the time dimension. We have 24 months of panel data -- throwing away the before/after structure wastes information and misses time-varying confounders that differencing removes. |
| **Synthetic Control** | Designed for a **single treated unit** where you construct one synthetic counterfactual. We have **10 treated regions** -- DiD handles the multi-unit case directly without needing to construct 10 separate synthetic controls. |
| **ITS** | ITS models a single unit's time series and extrapolates its pre-trend. We have a **natural control group** (10 untreated regions), which makes DiD strictly more credible -- the control group accounts for common shocks that ITS cannot distinguish from treatment effects. |
| **A/B test** | The rollout was a **business decision by region**, not randomized at the user level. We cannot retroactively randomize, so we work with the quasi-experiment the rollout created. |

**DiD wins because**: We have multiple treated units, multiple control units, multiple time periods, and a clean intervention date. This is the textbook DiD setting.

## Step 4: Check Parallel Trends

**WHY this matters**: Parallel trends is the **core identifying assumption** of DiD. It says that in the absence of treatment, the treated and control groups would have followed the same trajectory. If pre-treatment trends diverged, then the post-treatment divergence we attribute to the intervention could instead be a continuation of a pre-existing gap.

**WHAT WOULD GO WRONG if violated**: If treated regions were already trending upward faster than control regions before month 12, DiD would overestimate the treatment effect. The estimated effect would capture both the true causal impact and the pre-existing divergence.

**How we check**:
1. **Visual**: Plot pre-treatment trends for both groups. They should move roughly in parallel.
2. **Formal test**: Regress watch hours on month-by-treatment interactions for the pre-period. The interaction coefficients should be statistically indistinguishable from zero.

In [ ]:
# Visual check: pre-treatment trends
pre_data = monthly_means[monthly_means["month"] <= treatment_month]

fig, ax = plt.subplots(figsize=(10, 6))
for grp, label, color in [(1, "Treated regions", "#e74c3c"), (0, "Control regions", "#3498db")]:
    data = pre_data[pre_data["treated"] == grp]
    ax.plot(data["month"], data["watch_hours"], marker="o", label=label, color=color, linewidth=2)

ax.set_xlabel("Month")
ax.set_ylabel("Average Daily Watch Hours")
ax.set_title("Pre-Treatment Trends: Are They Parallel?")
ax.legend()
plt.tight_layout()
plt.show()

# Formal test: pre-period interaction terms
pre_df = df[df["post"] == 0].copy()
pre_df["month_x_treated"] = pre_df["month"] * pre_df["treated"]

pre_test = smf.ols("watch_hours ~ month + treated + month_x_treated", data=pre_df).fit(
    cov_type="cluster", cov_kwds={"groups": pre_df["region"]}
)
print("Formal parallel trends test (pre-period only):")
print(f"  month x treated coefficient: {pre_test.params['month_x_treated']:.4f}")
print(f"  p-value: {pre_test.pvalues['month_x_treated']:.4f}")
print()
if pre_test.pvalues["month_x_treated"] > 0.05:
    print("  => Cannot reject parallel trends (p > 0.05). Good -- DiD assumption is supported.")
else:
    print("  => Parallel trends rejected (p <= 0.05). DiD may be invalid.")

## Step 5: Estimate DiD with Regression

**WHY regression**: The 2x2 DiD formula works for a simple case, but regression handles the full panel -- multiple regions, multiple time periods, region fixed effects (absorbing permanent differences), and month fixed effects (absorbing common shocks).

**The model**:
```
watch_hours = region_FE + month_FE + β * (treated × post) + ε
```

The coefficient on the **interaction term** `treated × post` **IS** the DiD estimate. It captures the differential change in watch hours for treated regions after the intervention, relative to what control regions experienced -- exactly the double difference.

We cluster standard errors at the region level because the treatment is assigned at the region level and outcomes within a region are correlated over time.

In [ ]:
# Manual 2x2 DiD
mean_treated_pre = df[(df["treated"] == 1) & (df["post"] == 0)]["watch_hours"].mean()
mean_treated_post = df[(df["treated"] == 1) & (df["post"] == 1)]["watch_hours"].mean()
mean_control_pre = df[(df["treated"] == 0) & (df["post"] == 0)]["watch_hours"].mean()
mean_control_post = df[(df["treated"] == 0) & (df["post"] == 1)]["watch_hours"].mean()

did_manual = (mean_treated_post - mean_treated_pre) - (mean_control_post - mean_control_pre)

print("=== Manual 2x2 DiD ===")
print(f"  Treated:  pre = {mean_treated_pre:.3f},  post = {mean_treated_post:.3f},  change = {mean_treated_post - mean_treated_pre:.3f}")
print(f"  Control:  pre = {mean_control_pre:.3f},  post = {mean_control_post:.3f},  change = {mean_control_post - mean_control_pre:.3f}")
print(f"  DiD estimate = {did_manual:.3f}")
print(f"  True effect  = {true_effect:.3f}")
print()

# Regression DiD with fixed effects
did_model = smf.ols(
    "watch_hours ~ C(region) + C(month) + treated_post",
    data=df
).fit(cov_type="cluster", cov_kwds={"groups": df["region"]})

print("=== Regression DiD (with region and month fixed effects) ===")
print(f"  DiD coefficient (treated x post): {did_model.params['treated_post']:.4f}")
print(f"  Standard error:                   {did_model.bse['treated_post']:.4f}")
print(f"  t-statistic:                      {did_model.tvalues['treated_post']:.4f}")
print(f"  p-value:                           {did_model.pvalues['treated_post']:.6f}")
ci = did_model.conf_int().loc["treated_post"]
print(f"  95% CI:                            [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"  True effect:                       {true_effect:.4f}")
print()
print(f"  => The DiD estimate recovers an effect close to the true {true_effect} hrs/day.")

## Step 6: Event Study

**WHY**: The basic DiD gives a single average post-treatment effect. The event study goes further by estimating **how the effect evolves over time** -- separately for each month relative to the intervention.

**What we expect**:
- **Pre-treatment coefficients** (months before the launch) should be centered on **zero**. This is an additional validation of parallel trends: if the treatment group was already diverging before the intervention, these coefficients would be non-zero.
- **Post-treatment coefficients** (months after the launch) should be **positive and stable** around the true effect of 0.8 hrs/day.

The classic event study plot -- coefficients with 95% confidence intervals over event time -- is one of the most informative diagnostics in causal inference.

In [ ]:
# Create relative time variable (event time centered on treatment)
df["rel_month"] = df["month"] - treatment_month

# Drop the period just before treatment (rel_month = 0) as the reference
event_df = df.copy()
event_df["rel_month_factor"] = event_df["rel_month"].astype(str)

# Create interaction dummies for each relative month x treated
rel_months_sorted = sorted(df["rel_month"].unique())
ref_period = 0  # last pre-treatment period as reference

for rm in rel_months_sorted:
    if rm == ref_period:
        continue
    event_df[f"rm_{rm}"] = ((event_df["rel_month"] == rm) & (event_df["treated"] == 1)).astype(int)

interaction_cols = [f"rm_{rm}" for rm in rel_months_sorted if rm != ref_period]
formula = "watch_hours ~ C(region) + C(month) + " + " + ".join(interaction_cols)

event_model = smf.ols(formula, data=event_df).fit(
    cov_type="cluster", cov_kwds={"groups": event_df["region"]}
)

# Extract event study coefficients
event_coefs = []
for rm in rel_months_sorted:
    if rm == ref_period:
        event_coefs.append({"rel_month": rm, "coef": 0, "ci_low": 0, "ci_high": 0})
    else:
        col = f"rm_{rm}"
        ci = event_model.conf_int().loc[col]
        event_coefs.append({
            "rel_month": rm,
            "coef": event_model.params[col],
            "ci_low": ci[0],
            "ci_high": ci[1],
        })

event_coefs_df = pd.DataFrame(event_coefs)

# Event study plot
fig, ax = plt.subplots(figsize=(12, 6))
pre = event_coefs_df[event_coefs_df["rel_month"] <= 0]
post = event_coefs_df[event_coefs_df["rel_month"] > 0]

ax.errorbar(pre["rel_month"], pre["coef"],
            yerr=[pre["coef"] - pre["ci_low"], pre["ci_high"] - pre["coef"]],
            fmt="o", color="#3498db", capsize=3, linewidth=1.5, markersize=6, label="Pre-treatment")
ax.errorbar(post["rel_month"], post["coef"],
            yerr=[post["coef"] - post["ci_low"], post["ci_high"] - post["coef"]],
            fmt="o", color="#e74c3c", capsize=3, linewidth=1.5, markersize=6, label="Post-treatment")

ax.axhline(y=0, color="black", linestyle="-", linewidth=0.8)
ax.axhline(y=true_effect, color="green", linestyle="--", linewidth=1, alpha=0.7, label=f"True effect ({true_effect})")
ax.axvline(x=0.5, color="gray", linestyle="--", linewidth=1.5, alpha=0.5)
ax.set_xlabel("Months Relative to Treatment")
ax.set_ylabel("Estimated Effect (hrs/day)")
ax.set_title("Event Study: Dynamic Treatment Effects")
ax.legend()
plt.tight_layout()
plt.show()

print("Pre-treatment coefficients (should be ~0):")
for _, row in event_coefs_df[event_coefs_df["rel_month"] < 0].iterrows():
    print(f"  t = {row['rel_month']:+.0f}: {row['coef']:.4f}  [{row['ci_low']:.4f}, {row['ci_high']:.4f}]")
print()
print("Post-treatment coefficients (should be ~0.8):")
for _, row in event_coefs_df[event_coefs_df["rel_month"] > 0].iterrows():
    print(f"  t = {row['rel_month']:+.0f}: {row['coef']:.4f}  [{row['ci_low']:.4f}, {row['ci_high']:.4f}]")

## Step 7: Robustness Checks

**WHY**: Every causal estimate needs stress-testing. A single regression producing a statistically significant coefficient is not enough -- we need to verify the result is robust to alternative specifications and falsification tests.

**WHAT we test**:
1. **Placebo test**: Pretend the treatment happened at **month 6** instead of month 12, using only pre-treatment data. If DiD is working correctly, the placebo effect should be near zero -- there was no actual treatment at month 6.
2. **Varying the post-period window**: Instead of using all 12 post-treatment months, estimate the effect using only the first 3, 6, or 9 months. Consistent estimates across windows suggest the result is not driven by a specific time period.

In [ ]:
# === Placebo test: fake treatment at month 6 ===
placebo_month = 6
placebo_df = df[df["month"] <= treatment_month].copy()
placebo_df["placebo_post"] = (placebo_df["month"] > placebo_month).astype(int)
placebo_df["placebo_treated_post"] = placebo_df["treated"] * placebo_df["placebo_post"]

placebo_model = smf.ols(
    "watch_hours ~ C(region) + C(month) + placebo_treated_post",
    data=placebo_df
).fit(cov_type="cluster", cov_kwds={"groups": placebo_df["region"]})

print("=== Placebo Test (fake treatment at month 6) ===")
print(f"  Placebo DiD coefficient: {placebo_model.params['placebo_treated_post']:.4f}")
print(f"  p-value:                 {placebo_model.pvalues['placebo_treated_post']:.4f}")
if abs(placebo_model.params["placebo_treated_post"]) < 0.2 and placebo_model.pvalues["placebo_treated_post"] > 0.05:
    print("  => Placebo effect is near zero and not significant. Good -- supports causal interpretation.")
else:
    print("  => Warning: placebo effect is non-negligible. Investigate further.")
print()

# === Varying post-period window ===
print("=== Robustness: Varying Post-Period Window ===")
for window in [3, 6, 9, 12]:
    window_df = df[(df["month"] <= treatment_month) | (df["month"] <= treatment_month + window)].copy()
    window_model = smf.ols(
        "watch_hours ~ C(region) + C(month) + treated_post",
        data=window_df
    ).fit(cov_type="cluster", cov_kwds={"groups": window_df["region"]})
    ci = window_model.conf_int().loc["treated_post"]
    print(f"  Post window = {window:2d} months:  DiD = {window_model.params['treated_post']:.4f}  "
          f"95% CI [{ci[0]:.4f}, {ci[1]:.4f}]  p = {window_model.pvalues['treated_post']:.4f}")

print(f"\n  True effect: {true_effect:.4f}")
print("  => Consistent estimates across windows strengthen the causal claim.")

## Step 8: What Happens When Parallel Trends Fail

Everything above worked because our simulated data satisfied the parallel trends assumption. But what if it doesn't?

Here we simulate a scenario where **treated regions were already trending upward faster** than control regions *before* the intervention. This violates the core DiD assumption.

**What goes wrong**: DiD attributes the entire post-treatment divergence to the treatment, but part of that divergence was going to happen anyway. The result is a **biased, inflated estimate** -- we think the algorithm had a bigger effect than it actually did.

In [ ]:
# Simulate data where parallel trends are violated
rows_bad = []
for region in range(n_regions):
    is_treated = 1 if region in treated_regions else 0
    # Treated regions have a STEEPER pre-existing trend
    trend_slope = 0.10 if is_treated else 0.05
    for month in range(1, n_months + 1):
        is_post = 1 if month > treatment_month else 0
        watch_hours = (
            region_baselines[region]
            + trend_slope * month
            + true_effect * is_treated * is_post
            + np.random.normal(0, 0.3)
        )
        rows_bad.append({
            "region": f"R{region:02d}",
            "month": month,
            "treated": is_treated,
            "post": is_post,
            "watch_hours": watch_hours,
        })

df_bad = pd.DataFrame(rows_bad)
df_bad["treated_post"] = df_bad["treated"] * df_bad["post"]

# Visualize the violation
monthly_bad = df_bad.groupby(["month", "treated"])["watch_hours"].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: original data (parallel trends hold)
for grp, label, color in [(1, "Treated", "#e74c3c"), (0, "Control", "#3498db")]:
    data = monthly_means[monthly_means["treated"] == grp]
    axes[0].plot(data["month"], data["watch_hours"], marker="o", label=label, color=color, linewidth=2, markersize=4)
axes[0].axvline(x=treatment_month + 0.5, color="gray", linestyle="--", linewidth=1.5)
axes[0].set_title("Parallel Trends Hold (DiD Valid)")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Avg Watch Hours")
axes[0].legend()

# Right: bad data (parallel trends violated)
for grp, label, color in [(1, "Treated", "#e74c3c"), (0, "Control", "#3498db")]:
    data = monthly_bad[monthly_bad["treated"] == grp]
    axes[1].plot(data["month"], data["watch_hours"], marker="o", label=label, color=color, linewidth=2, markersize=4)
axes[1].axvline(x=treatment_month + 0.5, color="gray", linestyle="--", linewidth=1.5)
axes[1].set_title("Parallel Trends VIOLATED (DiD Biased)")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Avg Watch Hours")
axes[1].legend()

plt.tight_layout()
plt.show()

# Estimate DiD on the bad data
bad_model = smf.ols(
    "watch_hours ~ C(region) + C(month) + treated_post",
    data=df_bad
).fit(cov_type="cluster", cov_kwds={"groups": df_bad["region"]})

print("=== DiD on Data with Violated Parallel Trends ===")
print(f"  DiD estimate: {bad_model.params['treated_post']:.4f}")
print(f"  True effect:  {true_effect:.4f}")
print(f"  Bias:         {bad_model.params['treated_post'] - true_effect:.4f}")
print()
print("  => The DiD estimate is INFLATED because it attributes the pre-existing")
print("     faster trend in treated regions to the treatment.")
print()
print("  Lesson: ALWAYS check parallel trends. If they fail, consider:")
print("  - Synthetic Control (data-driven counterfactual)")
print("  - Matched DiD (PSM + DiD on matched sample)")
print("  - Conditioning on region-specific trends")

## Key Takeaways

### When to use DiD
- You have **panel data** (units observed over time)
- There is a **clear intervention point**
- Both **treated and control groups** are available
- Pre-treatment trends are **approximately parallel**

### Key assumptions
1. **Parallel trends** -- the treated group would have followed the same trajectory as the control group absent treatment
2. **No anticipation** -- treated units didn't change behavior before the intervention
3. **No spillover** -- control units are unaffected by the treatment
4. **Stable composition** -- group membership doesn't change over time

### What can go wrong
- **Parallel trends violated**: The most common failure. Pre-existing divergence gets attributed to the treatment, inflating the estimate. Always run event studies and placebo tests.
- **Anticipation effects**: If subjects change behavior before the formal intervention (e.g., regions prepare for the new algorithm), pre-period data is contaminated.
- **Spillover**: If control regions respond to what's happening in treated regions, the control counterfactual is compromised.

### Connection to other methods
- **PSM + DiD**: Match on pre-treatment characteristics, then apply DiD. Handles selection bias AND time trends.
- **Event study**: Extends DiD to show the dynamic treatment effect over time. Essential for validating parallel trends.
- **Staggered DiD**: When treatment timing varies across units, use Callaway-Sant'Anna or Sun-Abraham estimators to avoid the biases of naive two-way fixed effects.
- **Synthetic Control**: When parallel trends are implausible, Synthetic Control constructs a data-driven counterfactual from untreated units.